# Wikipedia agent

Goal: an agent that searches Wikipedia and fetches page content to answer questions.

## Load the API key

Walks up from this folder to the `.env` at the repo root, so it works at any depth.

In [ ]:
from dotenv import load_dotenv, find_dotenv

# walk up from this notebook's folder until a .env is found
env_path = find_dotenv(usecwd=True)
assert env_path, ".env not found in this folder or any parent"
load_dotenv(env_path)

## Request headers

Wikipedia blocks requests that don't send a `User-Agent`. Both tools will pass this same dict, so it lives in one place.

In [ ]:
HEADERS = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

# Search tool

Goal: a `search_wikipedia` function that returns the pages matching a query.

## Look at the raw response

Before writing the function, one plain request to see what the search API sends back. `params` builds the query string for us, so spaces and special characters are encoded.

In [ ]:
import requests

response = requests.get(
    "https://en.wikipedia.org/w/api.php",
    params={
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": "capybara",
    },
    headers=HEADERS,
)
response.status_code

In [ ]:
data = response.json()
data.keys()

## Walk into the JSON

The results sit under `query`. Two different numbers could answer "how many results", so we look at both.

In [ ]:
data["query"].keys()

In [ ]:
data["query"]["searchinfo"]

In [ ]:
len(data["query"]["search"])

In [ ]:
data["continue"]

## The search function

The same request, with the search term as an argument. It returns only the list under `query.search`. The docstring and type hints matter later: PydanticAI builds the tool description from them.

In [ ]:
def search_wikipedia(query: str) -> list[dict]:
    """
    Search Wikipedia for pages related to a query.

    Args:
        query (str): The search term, e.g. "capybara".

    Returns:
        list[dict]: Matching pages. Each has a title, snippet and other fields.
    """
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
        },
        headers=HEADERS,
    )
    response.raise_for_status()
    return response.json()["query"]["search"]

## Test it

In [ ]:
results = search_wikipedia("capybara")
len(results)

**Result:** `search_wikipedia("capybara")` returned **10** results (Wikipedia reports 930 total hits, but returns 10 per request).

# Analyse the results

Goal: count how many result titles contain the word "capybara".

## Look at the titles

In [ ]:
[r["title"] for r in results]

## Count the matches

`.lower()` makes the match case-insensitive. Each `in` check is `True` or `False`, and `sum` adds up the `True`s.

In [ ]:
sum("capybara" in r["title"].lower() for r in results)

**Result:** **6** of the 10 titles contain "capybara" (Capybara, Lesser capybara, Capybara (disambiguation), Capybara Games, Capybara Party, Capybara (software)).

# Get-page tool

Goal: a `get_page` function that returns the raw content of a Wikipedia page as a string.

## The get-page function

Same idea as the search function, but a different endpoint: `index.php` with `action=raw` returns the page's wikitext as plain text, not JSON, so we read `.text`.

In [ ]:
def get_page(title: str) -> str:
    """
    Fetch the raw content of a Wikipedia page.

    Args:
        title (str): The exact page title, e.g. "Capybara".

    Returns:
        str: The page content as wikitext, or a message if the page doesn't exist.
    """
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={
            "title": title,
            "action": "raw",
        },
        headers=HEADERS,
    )
    if response.status_code == 404:
        return f"Page '{title}' not found. Use search_wikipedia to find the exact title."
    response.raise_for_status()
    return response.text

## Test it

In [ ]:
content = get_page("Capybara")
len(content)

In [ ]:
content[:500]

**Result:** `get_page("Capybara")` returned **44,771** characters.

## A page that doesn't exist

The agent will sometimes guess a wrong title. A crash would end its run, so a 404 now comes back as a message the model can read and act on (`get_page` was updated above). Re-run the `get_page` cell first, then this one.

In [ ]:
get_page("Capybaraa")

**Note:** the first characters are raw wikitext (`{{Short description|...}}`, `{{Speciesbox ...}}`, `<ref ...>`), not prose. The model reads it fine, but it spends tokens on the markup.

# The agent

Goal: give a PydanticAI `Agent` the two tools and instructions for using them.

## Agent instructions

Two cases: a URL means go straight to `get_page`; a question means search first, then fetch.

In [ ]:
instructions = """
You're a Wikipedia research assistant.

Answer the user's question using only content fetched with your tools.

- If the user gives a Wikipedia URL, take the page title from it (the part after /wiki/) and call get_page with that title.
- Otherwise, call search_wikipedia first to find relevant pages, then call get_page on the most relevant ones and read them before answering.
- If get_page says a page was not found, search again to find the exact title.
- Keep the answer concise and say which pages you used.
"""

## Create the agent

The tools are passed as plain functions. PydanticAI builds each tool's description from its name, type hints and docstring. The `openai:` prefix picks the provider; the key comes from the environment loaded at the top.

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    name="wikipedia",
    model="openai:gpt-4o-mini",
    instructions=instructions,
    tools=[search_wikipedia, get_page],
)

# Single page question

Goal: ask about a page by URL. The agent should pull the title out of the URL and call `get_page`.

## Ask the question

In [ ]:
question = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

`run` is async, so it gets awaited - Jupyter allows `await` directly in a cell.

In [ ]:
result = await agent.run(question)

## Read the answer

In [ ]:
print(result.output)

## Which tools did it call?

Each message holds parts; a `tool-call` part is the model asking for one of our functions, with the arguments it chose.

In [ ]:
for m in result.all_messages():
    for p in m.parts:
        if p.part_kind == "tool-call":
            print("TOOL CALL:", p.tool_name, p.args)

**Result:** the agent called `get_page` once with `{"title":"Capybara"}` (no search) and answered that the page is about the **capybara, the largest rodent species** (*Hydrochoerus hydrochaeris*), native to South America - its habitat, diet, social behaviour, conservation status and popularity as a pet and meme.

# Search then fetch

Goal: a question with no URL, so the agent has to search first and then fetch the pages it picks.

## Ask the question

In [ ]:
question2 = "What are the main threats to capybara populations?"

In [ ]:
result2 = await agent.run(question2)

## Read the answer

In [ ]:
print(result2.output)

## Count the tool calls

Collects every `tool-call` part in order. The count comes from the message history, not from what the model says it did.

In [ ]:
tool_calls = [
    p for m in result2.all_messages() for p in m.parts if p.part_kind == "tool-call"
]
for p in tool_calls:
    print(p.tool_name, p.args)
len(tool_calls)

**Result:** **2** tool calls, in this order:

1. `search_wikipedia` with `{"query":"Capybara"}`
2. `get_page` with `{"title":"Capybara"}`

The search query was just the animal's name, not the question, and the agent fetched only the main page. Its answer named four threats: hunting for meat and pelts, conflict with farmers, habitat loss, and predation by jaguars and caimans. It added that the capybara is not currently a threatened species. The model can choose a different number of calls on another run.